In [ ]:
import logging
logging.getLogger("snowflake.connector").setLevel(logging.ERROR)

from snowflake.snowpark.context import get_active_session
from snowflake.ml.registry import Registry

import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

from sklearn.model_selection import train_test_split

In [ ]:
# ================================
# SETUP
# ================================
session = get_active_session()

# Set database & schema
session.sql("USE DATABASE AI_CRISIS_LOANS").collect()
session.sql("USE SCHEMA LOAN_ANALYTICS").collect()

# ================================
# FEATURES & TARGET
# ================================
FEATURE_COLS = [
    "AGE",
    "ANNUAL_INCOME_USD",
    "INCOME_VOLATILITY_SCORE",
    "NET_WORTH_USD",
    "YEARS_IN_CURRENT_ROLE",
    "MONTHS_UNEMPLOYED_LAST3YR",
    "HAS_AI_CERT_FLAG",
    "HAS_COLLATERAL_FLAG",
    "DIGITAL_LITERACY_SCORE",
    "EDUCATION_SCORE",
    "SOCIAL_SUPPORT_SCORE_NUM"
]

TARGET_COL = "TARGET"
print("✅ Session initialized")

In [ ]:
# ================================
# LOAD DATA
# ================================
full_df = session.table("AI_CRISIS_LOANS.LOAN_ANALYTICS.LOAN_FEATURES_TABLE_CLEAN")

df = full_df.to_pandas()

# Clean column names
df.columns = [c.strip('"') for c in df.columns]

# Fill missing values
df = df.fillna(0)

print(f"Rows loaded: {len(df):,}")
print("✅ Data loaded and cleaned")

In [ ]:
# ================================
# TRAIN / TEST SPLIT (PANDAS)
# ================================
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df[TARGET_COL]
)

train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f"Train rows: {len(train_df):,}")
print(f"Test rows:  {len(test_df):,}")

In [ ]:
# ================================
# LOAD PIPELINE MODEL (LATEST)
# ================================
reg = Registry(session=session)

versions = reg.get_model("AI_CRISIS_DEFAULT_MODEL_PIPELINE").show_versions()

latest_version = versions.sort_values("created_on", ascending=False).iloc[0]["name"]

print(f"Using latest version: {latest_version}")

model = reg.get_model("AI_CRISIS_DEFAULT_MODEL_PIPELINE").version(latest_version)

print("✅ Latest pipeline model loaded!")
print(f"Model version: {model.version_name}")

In [ ]:
from snowflake.snowpark import functions as F

print("\nRunning predictions...")

# --------------------------------
# 1. LIMIT + RESET
# --------------------------------
test_df = test_df.head(30000).reset_index(drop=True)

test_sp = session.create_dataframe(test_df)

# --------------------------------
# 2. PREDICT LABELS
# --------------------------------
pred_df = model.run(
    test_sp.select(FEATURE_COLS),
    function_name="predict"
)

# ✅ FIX: fully quoted column name
pred_df = pred_df.select(F.col('"output_feature_0"'))

pred_pd = pred_df.to_pandas()
pred_pd.columns = [c.strip('"') for c in pred_pd.columns]

pred_labels = pred_pd["output_feature_0"].values


# --------------------------------
# 3. PREDICT PROBABILITIES
# --------------------------------
prob_df = model.run(
    test_sp.select(FEATURE_COLS),
    function_name="predict_proba"
)

# ✅ FIX: fully quoted column name
prob_df = prob_df.select(F.col('"output_feature_1"'))

prob_pd = prob_df.to_pandas()
prob_pd.columns = [c.strip('"') for c in prob_pd.columns]

pred_probs = prob_pd["output_feature_1"].values


# --------------------------------
# 4. DONE
# --------------------------------
print("✅ Predictions completed!")
print("Rows processed:", len(pred_labels))

In [ ]:
# ================================
# BUILD EVALUATION DATAFRAME
# ================================
eval_df = test_df.copy()

eval_df["PREDICTION"] = pred_labels
eval_df["PROBABILITY"] = pred_probs

# Drop rows safely AFTER alignment
eval_df = eval_df.dropna(subset=[TARGET_COL])

In [ ]:
# ================================
# EVALUATE MODEL
# ================================
print("\nEvaluating model...")

actual = eval_df[TARGET_COL].astype(int)

# ✅ Use threshold tuning instead of raw predict
threshold = 0.3
predicted = (eval_df["PROBABILITY"] > threshold).astype(int)

print(f"\nEvaluating on {len(eval_df):,} rows...")
print(f"Threshold used: {threshold}")

accuracy = (actual == predicted).mean()
print(f"Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

print("\nClassification Report:")
print(classification_report(actual, predicted, target_names=["No Default", "Default"]))

print("\nConfusion Matrix:")
print(confusion_matrix(actual, predicted))

In [ ]:
# ================================
# SAVE RESULTS
# ================================

threshold = 0.3   # 🔥 SAME as evaluation

save_df = eval_df[[
    "BORROWER_ID",
    TARGET_COL,
    "PROBABILITY"
]].copy()

# ✅ Recompute prediction using threshold
save_df["PREDICTION"] = (save_df["PROBABILITY"] > threshold).astype(int)

# Correct flag
save_df["CORRECT"] = (save_df[TARGET_COL] == save_df["PREDICTION"]).astype(int)

# Save to Snowflake
session.create_dataframe(save_df).write.mode("overwrite").save_as_table(
    "AI_CRISIS_LOANS.LOAN_ANALYTICS.LOAN_PREDICTIONS_PIPELINE"
)

print("\n✅ Predictions saved to LOAN_PREDICTIONS_PIPELINE")

session.table("AI_CRISIS_LOANS.LOAN_ANALYTICS.LOAN_PREDICTIONS_PIPELINE").show(10)